## Formulario correo

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from sqlalchemy import create_engine
from sqlalchemy import text

import numpy as np

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

engine_mysql = create_engine(
    f"mysql+pymysql://{user_envio}:{pwd_envio}@{server_envio}:{port_mysql}/{db_envio}"
)


In [2]:
filename='TARGET.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_target_desembolso = pd.read_csv(ruta_archivo,sep='|')

# df_target_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
# df_target_desembolso.rename(columns={'FECHA_DESEMBOLSOS': 'fecha_desembolso'}, inplace=True)
df_target_desembolso = df_target_desembolso[['DNI']].copy()
df_target_desembolso['target'] = 1

filename='ACUM_DESEM.txt'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_fugas = pd.read_csv(ruta_archivo,sep='|')
df_fugas = df_fugas[['DNI','MONTO','ASESOR','CANALVENTA']].copy()

df_desembolso = df_target_desembolso[['DNI']].drop_duplicates().merge(
    df_fugas[['DNI']].drop_duplicates(),
    on=['DNI'],
    how='inner'
)

df_desembolso = df_target_desembolso.drop_duplicates().merge(
    df_fugas.drop_duplicates(),
    on=['DNI'],
    how='inner'
)
df_desembolso['target'] = df_desembolso['target'].fillna(0).astype(int)
df_desembolso['fugas'] = (df_desembolso['target'] == 0).astype(int)
df_desembolso.rename(columns={'DNI': 'dni_cliente'}, inplace=True)

df_desembolso['dni_cliente'] = (
    df_desembolso['dni_cliente']
    .astype(str)
    .str.replace(r'\D', '', regex=True)   
    .replace('', pd.NA)                     
    .str.zfill(8)                           
)



In [3]:
query = f"""
	SELECT * FROM Alice.prospectos_envio_alfin 
    where estado='procesado'
	and DATE(fecha_envio)>='2026-07-01'
"""
df_prospectos_envio_alfin = pd.read_sql(query, engine_mysql)

query = f"""
	SELECT * FROM Alice.prospectos_correos_alfin 
    where estado='ENVIADO'
	and DATE(fecha_envio)>='2026-07-01'
"""
df_prospectos_correos_alfin = pd.read_sql(query, engine_mysql)


In [4]:
df_prospectos_correos_alfin['dia_ref'] = pd.to_datetime(
    df_prospectos_correos_alfin['fecha_envio']
).dt.date

df_prospectos_envio_alfin['dia_ref'] = pd.to_datetime(
    df_prospectos_envio_alfin['fecha_envio']
).dt.date

df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente','dia_ref'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente','dia_ref'])

df_seguimiento = df_prospectos_envio_alfin[['dni_cliente', 'dia_ref']].merge(
    df_prospectos_correos_alfin[['dni_cliente', 'dia_ref','celular']],
    on=['dni_cliente', 'dia_ref'],
    how='inner'
)

In [5]:
df_seguimiento_1 = df_seguimiento.merge(
    df_desembolso,
    on=['dni_cliente'],
    how='left'
)

df_seguimiento_1['target'] = df_seguimiento_1['target'].fillna(0).astype(int)
df_seguimiento_1['fugas'] = df_seguimiento_1['fugas'].fillna(0).astype(int)

In [6]:
filename='RetiroDefinitivo_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_def_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_BlackList.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_blacklist = pd.read_csv(ruta_archivo,sep='|')
filename='RetiroDeGestion_Telefonos.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_telf = pd.read_csv(ruta_archivo,sep='|')
filename='retiro_correo_alfin.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_retiro_correo = pd.read_csv(ruta_archivo,sep=';')

filename='desembolso.csv'
ruta_archivo = os.path.join(ruta_alfin, filename)
df_des = pd.read_csv(ruta_archivo,sep=';')

df_def_blacklist = df_def_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_blacklist = df_blacklist.rename(columns={'DNI': 'dni_cliente'})
df_telf= df_telf.rename(columns={'TELEFONO': 'celular'})
df_retiro_correo= df_retiro_correo.rename(columns={'DNI': 'dni_cliente'})


# Blacklists de DNI
# df6 = df_des.copy()
# df6["celular"] = None
# df6 = df6[["dni_cliente", "celular"]]

# Blacklists de DNI
df1 = df_def_blacklist.copy()
df1["celular"] = None
df1 = df1[["dni_cliente", "celular"]]

df2 = df_blacklist.copy()
df2["celular"] = None
df2 = df2[["dni_cliente", "celular"]]

# Blacklist de teléfonos
df3 = df_telf.copy()
df3["dni_cliente"] = None
df3 = df3[["dni_cliente", "celular"]]

# Archivo con DNI y celular
df4 = df_retiro_correo[["dni_cliente", "celular"]].copy()




In [7]:

# Unir todo
df_retiros = pd.concat(
    [df1, df2, df3, df4],
    ignore_index=True
)

dni_retiro = set(df_retiros['dni_cliente'].dropna())
cel_retiro = set(df_retiros['celular'].dropna())

df_seguimiento_1['retiro'] = (
    df_seguimiento_1['dni_cliente'].isin(dni_retiro) |
    df_seguimiento_1['celular'].isin(cel_retiro)
).astype(int)

C:\Users\DATA\AppData\Local\Temp\ipykernel_7988\2023645342.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_retiros = pd.concat(


### spark -- datos faltantes

In [8]:
ruta_archivo = os.path.join(ruta_csv, 'tmp_muestra_alfin.csv')
df_seguimiento_1['dni_cliente'].to_csv(ruta_archivo, index=False,sep=';')
# ruta_archivo = os.path.join(ruta_alfin, 'desembolso.csv')
# df_des.to_csv(ruta_archivo, index=False,sep=';')

In [9]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [10]:
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db.csv'
df_validar_01=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
filename='Consulta_de_Campañas_202607_V2_SS_EXT_CAMBIO_db2.csv'
df_validar_02=cargar_archivo_csv_ruta(spark,filename,';',True,ruta_alfin)
df_validar=df_validar_01.unionByName(df_validar_02)

filename='tmp_muestra_alfin.csv'
df_tmp_dni=cargar_archivo_csv(spark,filename,';',True)

filename='retiro_alfin_acum.csv'
df_tmp_retiro=cargar_archivo_csv(spark,filename,';',True)

In [11]:
# filename='Libro10.csv'
# aja=cargar_archivo_csv(spark,filename,';',True)
# aja = aja.withColumn(
#     "DNI",
#     F.right(
#         F.concat(F.lit("00000000"), F.col("DNI")),
#         F.lit(8)
#     )
# )
# overwrite_table_SQL(spark,aja,f'ALFIN_BORRAR_BORRAR_1',server_kishin,user_kishin,pwd_kishin,'DANTALION')

In [12]:

df_tmp_dni = df_tmp_dni.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("dni_cliente")),
        F.lit(8)
    )
)

df_validar = df_validar.withColumn(
    "dni_cliente",
    F.right(
        F.concat(F.lit("00000000"), F.col("DNI")),
        F.lit(8)
    )
).drop('DNI')

In [13]:
# df_tmp_retiro = df_tmp_retiro.withColumn(
#     "dni_cliente",
#     F.right(
#         F.concat(F.lit("00000000"), F.col("dni_cliente")),
#         F.lit(8)
#     )
# ).drop('DNI')

In [14]:
overwrite_table_SQL(spark,df_tmp_dni,f'tmp_muestra_cliente_alfin_borrar',server_kishin,user_kishin,pwd_kishin,'DANTALION')
# overwrite_table_SQL(spark,df_tmp_retiro,f'tmp_muestra_cliente_alfin_borrar_retiro_1',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [15]:
query = """
    select * from DANTALION.dbo.tmp_muestra_cliente_alfin_borrar 
    """
df_dni_ref=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_validar=df_validar.join(df_dni_ref,['dni_cliente'],'inner')

query = """
    select a.NUMERO_DOCUMENTO as dni_cliente
    from DANTALION.dbo.Base_Maestra_ALFIN_BK_Vigente a
    inner join tmp_muestra_cliente_alfin_borrar b
    on a.NUMERO_DOCUMENTO=b.dni_cliente
    """
df_ref_base=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_validar_pd = df_validar.toPandas()
df_ref_base_pd = df_ref_base.toPandas()

### completar base + columnas de consulta campaña

In [16]:
dni_en_base = set(df_ref_base_pd['dni_cliente'].dropna())

df_seguimiento_1['en_base'] = (
    df_seguimiento_1['dni_cliente'].isin(dni_en_base) 
).astype(int)

In [17]:
df = df_seguimiento_1.copy()


In [18]:
def contar_dias_sin_domingo(fecha_inicio, fecha_fin):
    """
    Cuenta los días transcurridos después de fecha_inicio
    hasta fecha_fin, incluyendo lunes a sábado y excluyendo domingo.

    Ejemplo:
    sábado -> lunes = 1 día
    viernes -> lunes = 2 días
    """

    if pd.isna(fecha_inicio) or pd.isna(fecha_fin):
        return np.nan

    fecha_inicio = pd.Timestamp(fecha_inicio).normalize()
    fecha_fin = pd.Timestamp(fecha_fin).normalize()

    if fecha_fin <= fecha_inicio:
        return 0

    return np.busday_count(
        fecha_inicio.date() + pd.Timedelta(days=1),
        fecha_fin.date() + pd.Timedelta(days=1),
        weekmask="1111110"  # lunes a sábado
    )



In [19]:
df.head()

,dni_cliente,dia_ref,celular,target,MONTO,ASESOR,CANALVENTA,fugas,retiro,en_base
0,19337433,2026-07-03,963517955,0,NaN,NaN,NaN,0,0,1
1,19998396,2026-07-03,937474638,0,NaN,NaN,NaN,0,0,1
2,40949266,2026-07-03,943587132,0,NaN,NaN,NaN,0,0,1
3,40319405,2026-07-03,923809089,0,NaN,NaN,NaN,0,0,1
4,00809150,2026-07-03,987302232,0,NaN,NaN,NaN,0,0,1


In [19]:

# ============================================================
# 3. CALCULAR INFORMACIÓN POR DNI
# ============================================================

resumen_dni = (
    df
    .groupby("dni_cliente", as_index=False)
    .agg(
        cantidad_registros=("dia_ref", "size"),
        q_envios=("dia_ref", "nunique")
    )
).copy()
fecha_hoy = pd.Timestamp.today().normalize()

In [20]:
df_ultimo = (
    df
    .sort_values(
        ["dni_cliente", "dia_ref"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset="dni_cliente",
        keep="first"
    )
).copy()


In [21]:
df_resultado = df_ultimo.merge(
    resumen_dni,
    on="dni_cliente",
    how="left"
)

df_resultado["dias_sin_domingo"] = (
    df_resultado["dia_ref"]
    .apply(
        lambda fecha: contar_dias_sin_domingo(
            fecha,
            fecha_hoy
        )
    )
)

In [22]:
import numpy as np

condiciones = [
    # Un solo envío
    (df_resultado["q_envios"] == 1) &
    (df_resultado["dias_sin_domingo"] == 5),

    (df_resultado["q_envios"] == 1) &
    (df_resultado["dias_sin_domingo"] ==6),

    (df_resultado["q_envios"] == 1) &
    (df_resultado["dias_sin_domingo"] < 5),

    # Más de un envío
    (df_resultado["q_envios"] > 1) &
    (df_resultado["dias_sin_domingo"] == 2),

    (df_resultado["q_envios"] > 1) &
    (df_resultado["dias_sin_domingo"] ==3),

    (df_resultado["q_envios"] > 1) &
    (df_resultado["dias_sin_domingo"] < 2)
]

resultados = [
    "ENVIAR",
    "CONSIDERAR1",
    "VIGENTE",
    "ENVIAR",
    "CONSIDERAR2",
    "VIGENTE"
]

df_resultado["estado_envio"] = np.select(
    condiciones,
    resultados,
    default="NO APLICA"
)

In [24]:
df_resultado.head()


,dni_cliente,dia_ref,celular,target,MONTO,ASESOR,CANALVENTA,fugas,retiro,en_base,cantidad_registros,q_envios,dias_sin_domingo,estado_envio
0,00001050,2026-07-25,980279060,0,NaN,NaN,NaN,0,0,0,1,1,4,VIGENTE
1,00009896,2026-07-08,960359161,0,NaN,NaN,NaN,0,0,0,1,1,19,NO APLICA
2,00013700,2026-07-15,939419006,0,NaN,NaN,NaN,0,0,0,2,2,13,NO APLICA
3,00015623,2026-07-25,969544482,0,NaN,NaN,NaN,0,0,0,1,1,4,VIGENTE
4,00016021,2026-07-08,948070815,0,NaN,NaN,NaN,0,0,0,1,1,19,NO APLICA


In [23]:
df_validar_pd_seleccion=df_validar_pd[['dni_cliente','COLOR_FINAL','USER_V3','FRESCURA','PROPENSION_DISTRIBUCION','OFERTA_MAX']]

In [24]:
oferta = pd.to_numeric(
    df_validar_pd_seleccion["OFERTA_MAX"],
    errors="coerce"
)

condiciones = [
    oferta.isna(),
    oferta < 2500,
    oferta < 5000,
    oferta < 7500,
    oferta < 10000,
    oferta < 12500,
    oferta < 15000,
    oferta < 17500,
    oferta < 20000,
    oferta < 22500,
    oferta < 25000,
    oferta < 27500,
]

valores = [
    "SIN DATO",
    "00.[0 - 2,500)",
    "01.[2,500 - 5,000)",
    "02.[5,000 - 7,500)",
    "03.[7,500 - 10,000)",
    "04.[10,000 - 12,500)",
    "05.[12,500 - 15,000)",
    "06.[15,000 - 17,500)",
    "07.[17,500 - 20,000)",
    "08.[20,000 - 22,500)",
    "09.[22,500 - 25,000)",
    "10.[25,000 - 27,500)",
]

df_validar_pd_seleccion["rango_OFERTA"] = np.select(
    condiciones,
    valores,
    default="11.[27,500 A MÁS]"
)

C:\Users\DATA\AppData\Local\Temp\ipykernel_7988\1526375628.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_validar_pd_seleccion["rango_OFERTA"] = np.select(


In [25]:
df_resumen_1=df_resultado.merge(df_validar_pd_seleccion,on='dni_cliente',how='left')
df_resumen_1 = df_resumen_1.drop_duplicates(subset=['dni_cliente'])


In [26]:
ruta_archivo = os.path.join(ruta_csv, 'tmp_resumen1.csv')
df_resumen_1.to_csv(ruta_archivo, index=False,sep=';')

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\DATA\\Documents\\datos\\05_subir_csv\\tmp_resumen1.csv'

In [27]:
# fechas = pd.to_datetime(['2026-07-28'])

df_resumen_1_2=df_resumen_1[
    (df_resumen_1['fugas'] == 0) &
    (df_resumen_1['retiro'] == 0) &
    # (df_resumen_1['dia_ref'].isin(fechas)) &
    (df_resumen_1['q_envios']!=1) &
    (df_resumen_1['estado_envio'].isin(['ENVIAR','CONSIDERAR1','CONSIDERAR2'])) &
    (df_resumen_1['MONTO'].isna()) 
].copy()
df_resumen_1_2.shape


(2136, 20)

In [36]:
df_resumen_1_2.groupby('q_envios').size().reset_index(name='cantidad')

,q_envios,cantidad
0,2,251
1,3,46
2,4,140
3,5,87
4,6,33
5,7,3


In [37]:
df_resumen_1.groupby(
    ['dia_ref', 'q_envios']
).size().reset_index(name='cantidad')

,dia_ref,q_envios,cantidad
0,2026-07-15,1,932
1,2026-07-15,2,458
2,2026-07-16,1,1708
3,2026-07-16,3,1


In [28]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)


In [69]:
df_resumen_1_2[df_resumen_1_2['MONTO'].isna()].head()

,dni_cliente,dia_ref,celular,target,MONTO,ASESOR,CANALVENTA,fugas,retiro,en_base,cantidad_registros,q_envios,dias_sin_domingo,estado_envio,COLOR_FINAL,USER_V3,FRESCURA,PROPENSION_DISTRIBUCION,OFERTA_MAX
6,00016025,2026-07-24,943203665,0,NaN,NaN,NaN,0,0,0,3,3,3,CONSIDERAR2,AMARILLO CLARO,4. MES + PLD No Peers,4,1,7600
11,00027081,2026-07-24,969804737,0,NaN,NaN,NaN,0,0,0,3,3,3,CONSIDERAR2,VERDE CLARO,7. Peers,4,1,13500
14,00032676,2026-07-25,994586314,0,NaN,NaN,NaN,0,0,0,2,2,2,ENVIAR,VERDE CLARO,3. MES + PLD Peers,1,1,11600
23,00048101,2026-07-25,997971765,0,NaN,NaN,NaN,0,0,0,5,5,2,ENVIAR,AMARILLO CLARO,14. Otros Bancarizados,3,2,16000
29,00051909,2026-07-24,944862382,0,NaN,NaN,NaN,0,0,0,3,3,3,CONSIDERAR2,AMARILLO CLARO,10. Dependiente + Convenios,4,1,5700


In [29]:
df_resumen_1_2=df_resumen_1_2.rename(columns={'COLOR_FINAL':'color'})

In [71]:
df_resumen_1_2.shape

(2936, 19)

In [30]:
df_prospectos_correos_alfin=df_prospectos_correos_alfin.drop(columns='color')


In [31]:
# df_prospectos_correos_alfin=df_prospectos_correos_alfin.drop(columns='color')
df_prospectos_envio_alfin=df_prospectos_envio_alfin.merge(df_resumen_1_2[['dni_cliente','color']],on='dni_cliente',how='inner')
df_prospectos_correos_alfin=df_prospectos_correos_alfin.merge(df_resumen_1_2[['dni_cliente','color']],on='dni_cliente',how='inner')

In [32]:
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente'])


In [33]:
df_prospectos_envio_alfin=df_prospectos_envio_alfin.drop(columns='color')

In [34]:
# df_prospectos_envio_alfin=df_prospectos_envio_alfin.merge(df_resumen_1[['dni_cliente','COLOR_FINAL']],on='dni_cliente',how='inner')
# df_prospectos_correos_alfin=df_prospectos_correos_alfin.merge(df_resumen_1[['dni_cliente','COLOR_FINAL']],on='dni_cliente',how='inner')

df_prospectos_correos_alfin=df_prospectos_correos_alfin[['canal_campo', 'supervisor', 'ejecutivo_target', 'codigo_ejecutivo_id', 'cdv_alfin_banco', 'dni_cliente', 'nombre_cliente', 'color', 'monto_solicitado', 'celular', 'agencia_atencion', 'fecha_visita','hora_visita']] .copy()
df_prospectos_correos_alfin['tipo_carga']='MANUAL'

df_prospectos_envio_alfin=df_prospectos_envio_alfin[['dni_vendedor', 'operador', 'dni_cliente', 'nombre_cliente', 'telefono_cliente', 'agencia_tienda', 'fecha_visita', 'monto_solicitado', 'tipo_gestion']].copy()
df_prospectos_correos_alfin['fecha_visita']='2026-07-24'
df_prospectos_envio_alfin['fecha_visita']='2026-07-24'

df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset='dni_cliente')
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset='dni_cliente')

display(df_prospectos_correos_alfin.head(2))
display(df_prospectos_envio_alfin.head(2))


,canal_campo,supervisor,ejecutivo_target,codigo_ejecutivo_id,cdv_alfin_banco,dni_cliente,nombre_cliente,color,monto_solicitado,celular,agencia_atencion,fecha_visita,hora_visita,tipo_carga
0,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,26700559,SORAIDA HUAMAN MORALES,NARANJA CLARO,1500.0,976567628,CAJAMARCA,2026-07-24,0 days,MANUAL
1,CALL CENTER / TARGET OUTSOURCING,CARLOS ENRIQUE RAMIREZ CACHIQUE,BOT,00000001,ROSA HONOR,71481682,AUGUSTO EMMANUEL APONTE CASTRO,AMARILLO OSCURO,3000.0,941730498,CASTILLA,2026-07-24,0 days,MANUAL


,dni_vendedor,operador,dni_cliente,nombre_cliente,telefono_cliente,agencia_tienda,fecha_visita,monto_solicitado,tipo_gestion
0,00000001,TARGET,16767136,FRANCISCO TIMANA BECERRA,954391558,737870 - VILLA MARIA 2,2026-07-24,7700,Derivacion
1,00000001,TARGET,47587005,LUIS ARNOLD YUPANQUI GUTIERREZ,963742019,734265 - TRUJ CENTRO,2026-07-24,14000,Derivacion


In [35]:
df_prospectos_correos_alfin.shape

(2136, 14)

In [36]:
df_prospectos_envio_alfin = df_prospectos_envio_alfin.drop_duplicates(subset=['dni_cliente'])
df_prospectos_correos_alfin = df_prospectos_correos_alfin.drop_duplicates(subset=['dni_cliente'])


In [37]:


df_prospectos_correos_alfin.to_sql(
    name="prospectos_correos_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

df_prospectos_envio_alfin.to_sql(
    name="prospectos_envio_alfin",
    con=engine_mysql,
    if_exists="append",
    index=False,
    chunksize=1000
)

C:\Users\DATA\AppData\Local\Temp\ipykernel_7988\3762344665.py:1: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  df_prospectos_correos_alfin.to_sql(


2136

In [164]:
query = f"""
		SELECT dni_cliente FROM Alice.prospectos_envio_alfin 
    where fecha_visita='2026-07-20'
"""
df_estan = pd.read_sql(query, engine_mysql)

In [165]:
df_resumen_1 = df_resumen_1[
    ~df_resumen_1['dni_cliente'].isin(df_estan['dni_cliente'])
].copy()
df_resumen_1.shape

(0, 23)